In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shxn65/fake-review-dataset/fake reviews dataset.csv


In [19]:
!pip install pandas scikit-learn textblob nltk

In [20]:
import pandas as pd
import numpy as np
import nltk
import re

from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score



In [21]:
df = pd.read_csv("/kaggle/input/datasets/shxn65/fake-review-dataset/fake reviews dataset.csv")

df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [22]:
df = df[["text_","rating","label"]]

df = df.dropna()

df.head()

,text_,rating,label
0,"Love this! Well made, sturdy, and very comfor...",5.0,CG
1,"love it, a great upgrade from the original. I...",5.0,CG
2,This pillow saved my back. I love the look and...,5.0,CG
3,"Missing information on how to use it, but it i...",1.0,CG
4,Very nice set. Good quality. We have had the s...,5.0,CG


In [23]:
df.columns

Index(['text_', 'rating', 'label'], dtype='object')

In [24]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]','',text)
    return text

df["clean_review"] = df["text_"].apply(clean_text)

In [25]:
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df["sentiment"] = df["clean_review"].apply(get_sentiment)

In [26]:
promo_words = ["best","amazing","must buy","highly recommend","excellent"]

def promo_score(text):
    score = 0
    for word in promo_words:
        if word in text:
            score += 1
    return score

df["promo_score"] = df["clean_review"].apply(promo_score)

In [27]:
vectorizer = TfidfVectorizer(max_features=3000)

X_text = vectorizer.fit_transform(df["clean_review"]).toarray()

In [28]:
extra_features = df[["sentiment","promo_score"]].values

X = np.concatenate((X_text,extra_features),axis=1)

y = df["label"]

In [29]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = RandomForestClassifier()

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

Accuracy: 0.8788178558179794


In [30]:
prob = model.predict_proba(X)

df["fake_probability"] = prob[:,1]

df["TrustScore"] = 100 - (df["fake_probability"]*100)

df.head()

,text_,rating,label,clean_review,sentiment,promo_score,fake_probability,TrustScore
0,"Love this! Well made, sturdy, and very comfor...",5.0,CG,love this well made sturdy and very comfortab...,0.442500,0,0.12,88.0
1,"love it, a great upgrade from the original. I...",5.0,CG,love it a great upgrade from the original ive...,0.558333,0,0.26,74.0
2,This pillow saved my back. I love the look and...,5.0,CG,this pillow saved my back i love the look and ...,0.250000,0,0.11,89.0
3,"Missing information on how to use it, but it i...",1.0,CG,missing information on how to use it but it is...,0.300000,0,0.19,81.0
4,Very nice set. Good quality. We have had the s...,5.0,CG,very nice set good quality we have had the set...,0.740000,0,0.09,91.0


In [31]:
def check_review(review):

    clean = clean_text(review)

    sentiment = TextBlob(clean).sentiment.polarity

    promo = promo_score(clean)

    text_vector = vectorizer.transform([clean]).toarray()

    features = np.concatenate((text_vector,[[sentiment,promo]]),axis=1)

    prob = model.predict_proba(features)[0][1]

    trust = 100 - (prob*100)

    print("Review:",review)
    print("TrustScore:",round(trust,2))

    if trust < 50:
        print("⚠ Likely Fake Review")
    else:
        print("Genuine Review")

In [32]:
def check_review(review):

    clean = clean_text(review)

    sentiment = TextBlob(clean).sentiment.polarity

    promo = promo_score(clean)

    text_vector = vectorizer.transform([clean]).toarray()

    features = np.concatenate((text_vector, [[sentiment, promo]]), axis=1)

    prob = model.predict_proba(features)[0][1]

    trust = 100 - (prob * 100)

    print("Review:", review)
    print("TrustScore:", round(trust,2))

    # Classification
    if trust < 50:
        print("⚠ Likely Fake Review")
    else:
        print("Genuine Review")

    # EXPLANATION PART
    print("\n--- Explanation ---")

    if promo > 0:
        print("⚠ Promotional language detected")

    if sentiment < 0:
        print("⚠ Negative sentiment detected")

In [33]:
import ipywidgets as widgets
from IPython.display import display

review_input = widgets.Text(
    placeholder="Enter review here",
    description="Review:",
    layout=widgets.Layout(width='500px')
)

button = widgets.Button(description="Check Review")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        check_review(review_input.value)

button.on_click(on_click)

display(review_input, button, output)

Text(value='', description='Review:', layout=Layout(width='500px'), placeholder='Enter review here')

Button(description='Check Review', style=ButtonStyle())

Output()

In [34]:
check_review("Amazing product must buy highly recommend")

Review: Amazing product must buy highly recommend
TrustScore: 31.0
⚠ Likely Fake Review

--- Explanation ---
⚠ Promotional language detected
